# Hybrid Book Recommendation System Using Collaborative Filtering and Content-Based Filtering     

## Objectives

- Build a collaborative filtering model using SVD/ALS
- Build a content-based filtering model using TF-IDF and cosine similarity
- Combine both models into a hybrid recommendation system
- Handle cold-start problem for new users
- Evaluate model performance using ranking metrics

In [5]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import sklearn
import warnings; warnings.simplefilter('ignore')

##   1)Loading Book Data:

In [6]:
book_data = pd.read_csv(r"../data/raw/Book_data.csv")
book_data.head()

,Id,Name,Authors,ISBN,Rating,PublishYear,PublishMonth,PublishDay,Publisher,RatingDist5,RatingDist4,RatingDist3,RatingDist2,RatingDist1,RatingDistTotal,CountsOfReview,Language,pagesNumber,Description,Count of text reviews
0,1000000,Flight from Eden,Kathryn A. Graham,0595199402,4.00,2001,1,10,Writer's Showcase Press,5:1,4:1,3:1,2:0,1:0,total:3,1,NaN,380,"What could a computer expert, a mercenary with...",1
1,1000001,Roommates Again,Kathryn O. Galbraith,0689505973,3.20,1994,1,4,Margaret K. McElderry Books,5:0,4:3,3:1,2:0,1:1,total:5,1,NaN,44,"During their stay at Camp Sleep-Away, sisters ...",1
2,1000003,The King At The Door,Brock Cole,0374440417,3.95,1992,31,12,Farrar Straus Giroux,5:5,4:9,3:4,2:1,1:0,total:19,0,NaN,32,A poorly dressed old man appears at an inn and...,0
3,1000004,"Giotto: The Scrovegni Chapel, Padua",Bruce Cole,080761310X,4.47,1993,1,8,George Braziller,5:9,4:5,3:0,2:1,1:0,total:15,2,NaN,118,This beautiful series lavishly illustrates the...,2
4,1000005,Larky Mavis,Brock Cole,0374343659,3.69,2001,3,8,"Farrar, Straus and Giroux (BYR)",5:19,4:12,3:9,2:7,1:4,total:51,8,NaN,32,<b>Another orginal picture-book fairy tale</b>...,8


In [7]:
book_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39705 entries, 0 to 39704
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Id                     39705 non-null  int64  
 1   Name                   39705 non-null  object 
 2   Authors                39705 non-null  object 
 3   ISBN                   39577 non-null  object 
 4   Rating                 39705 non-null  float64
 5   PublishYear            39705 non-null  int64  
 6   PublishMonth           39705 non-null  int64  
 7   PublishDay             39705 non-null  int64  
 8   Publisher              39345 non-null  object 
 9   RatingDist5            39705 non-null  object 
 10  RatingDist4            39705 non-null  object 
 11  RatingDist3            39705 non-null  object 
 12  RatingDist2            39705 non-null  object 
 13  RatingDist1            39705 non-null  object 
 14  RatingDistTotal        39705 non-null  object 
 15  Co

In [8]:
book_data.shape

(39705, 20)

## Loading Ratings Data

In [9]:
ratings_data = pd.read_csv(r"../data/raw/book_rating.csv")
ratings_data.head()

,user_id,book_id,rating
0,U0058,B0007,3
1,U0053,B0174,5
2,U0017,B0008,1
3,U0014,B0144,2
4,U0215,B0057,4


In [10]:
ratings_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1034 entries, 0 to 1033
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  1034 non-null   object
 1   book_id  1034 non-null   object
 2   rating   1034 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 24.4+ KB


In [11]:
ratings_data.shape

(1034, 3)

##  2)EDA OF Rating_DATA:

In [12]:
ratings_data.head()

,user_id,book_id,rating
0,U0058,B0007,3
1,U0053,B0174,5
2,U0017,B0008,1
3,U0014,B0144,2
4,U0215,B0057,4


In [13]:
ratings_data.dtypes

user_id    object
book_id    object
rating      int64
dtype: object

In [14]:
ratings_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1034 entries, 0 to 1033
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  1034 non-null   object
 1   book_id  1034 non-null   object
 2   rating   1034 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 24.4+ KB


In [15]:
ratings_data.isnull().sum()

user_id    0
book_id    0
rating     0
dtype: int64

In [16]:
print("Unique Users:", ratings_data['user_id'].nunique())
print("Unique Books:", ratings_data['book_id'].nunique())

Unique Users: 291
Unique Books: 199


In [17]:
ratings_data['rating'].value_counts().sort_index()

rating
1    197
2    204
3    222
4    203
5    208
Name: count, dtype: int64

## EDA of Book_Data

In [ ]:
ratings_data['rating'].value_counts().sort_index()

,Id,Name,Authors,ISBN,Rating,PublishYear,PublishMonth,PublishDay,Publisher,RatingDist5,RatingDist4,RatingDist3,RatingDist2,RatingDist1,RatingDistTotal,CountsOfReview,Language,pagesNumber,Description,Count of text reviews
0,1000000,Flight from Eden,Kathryn A. Graham,0595199402,4.00,2001,1,10,Writer's Showcase Press,5:1,4:1,3:1,2:0,1:0,total:3,1,NaN,380,"What could a computer expert, a mercenary with...",1
1,1000001,Roommates Again,Kathryn O. Galbraith,0689505973,3.20,1994,1,4,Margaret K. McElderry Books,5:0,4:3,3:1,2:0,1:1,total:5,1,NaN,44,"During their stay at Camp Sleep-Away, sisters ...",1
2,1000003,The King At The Door,Brock Cole,0374440417,3.95,1992,31,12,Farrar Straus Giroux,5:5,4:9,3:4,2:1,1:0,total:19,0,NaN,32,A poorly dressed old man appears at an inn and...,0
3,1000004,"Giotto: The Scrovegni Chapel, Padua",Bruce Cole,080761310X,4.47,1993,1,8,George Braziller,5:9,4:5,3:0,2:1,1:0,total:15,2,NaN,118,This beautiful series lavishly illustrates the...,2
4,1000005,Larky Mavis,Brock Cole,0374343659,3.69,2001,3,8,"Farrar, Straus and Giroux (BYR)",5:19,4:12,3:9,2:7,1:4,total:51,8,NaN,32,<b>Another orginal picture-book fairy tale</b>...,8


In [19]:
book_data.dtypes

Id                         int64
Name                      object
Authors                   object
ISBN                      object
Rating                   float64
PublishYear                int64
PublishMonth               int64
PublishDay                 int64
Publisher                 object
RatingDist5               object
RatingDist4               object
RatingDist3               object
RatingDist2               object
RatingDist1               object
RatingDistTotal           object
CountsOfReview             int64
Language                  object
pagesNumber                int64
Description               object
Count of text reviews      int64
dtype: object

In [20]:
book_data.isnull().sum()

Id                           0
Name                         0
Authors                      0
ISBN                       128
Rating                       0
PublishYear                  0
PublishMonth                 0
PublishDay                   0
Publisher                  360
RatingDist5                  0
RatingDist4                  0
RatingDist3                  0
RatingDist2                  0
RatingDist1                  0
RatingDistTotal              0
CountsOfReview               0
Language                 32692
pagesNumber                  0
Description               5146
Count of text reviews        0
dtype: int64

In [21]:
print("Unique Books:", book_data['Id'].nunique())
print("Unique Authors:", book_data['Authors'].nunique())
print("Unique Publishers:", book_data['Publisher'].nunique())

Unique Books: 39705
Unique Authors: 28564
Unique Publishers: 7474


In [22]:
print("Missing Descriptions:", book_data['Description'].isnull().sum())


Missing Descriptions: 5146


## 3)Data Preparation (Data Cleaning and Feature Engineering)

### Book Data processing (For Content Base Filtering)

####  Handle missing values

In [23]:
book_data['Description'] = book_data['Description'].fillna('')
book_data['Publisher'] = book_data['Publisher'].fillna('Unknown')
book_data['ISBN'] = book_data['ISBN'].fillna('Unknown')


In [24]:
book_data['content'] = (
    book_data['Name'] + " " +
    book_data['Authors'] + " " +
    book_data['Description']
)

In [ ]:
book_data[['Name', 'Authors', 'content']].head()

,Name,Authors,content
0,Flight from Eden,Kathryn A. Graham,Flight from Eden Kathryn A. Graham What could ...
1,Roommates Again,Kathryn O. Galbraith,Roommates Again Kathryn O. Galbraith During th...
2,The King At The Door,Brock Cole,The King At The Door Brock Cole A poorly dress...
3,"Giotto: The Scrovegni Chapel, Padua",Bruce Cole,"Giotto: The Scrovegni Chapel, Padua Bruce Cole..."
4,Larky Mavis,Brock Cole,Larky Mavis Brock Cole <b>Another orginal pict...


In [26]:
book_data.loc[:, ['Name', 'Authors', 'Description']].head()

,Name,Authors,Description
0,Flight from Eden,Kathryn A. Graham,"What could a computer expert, a mercenary with..."
1,Roommates Again,Kathryn O. Galbraith,"During their stay at Camp Sleep-Away, sisters ..."
2,The King At The Door,Brock Cole,A poorly dressed old man appears at an inn and...
3,"Giotto: The Scrovegni Chapel, Padua",Bruce Cole,This beautiful series lavishly illustrates the...
4,Larky Mavis,Brock Cole,<b>Another orginal picture-book fairy tale</b>...


In [27]:
book_data.loc[book_data['Rating'] > 4].head()

,Id,Name,Authors,ISBN,Rating,PublishYear,PublishMonth,PublishDay,Publisher,RatingDist5,...,RatingDist3,RatingDist2,RatingDist1,RatingDistTotal,CountsOfReview,Language,pagesNumber,Description,Count of text reviews,content
3,1000004,"Giotto: The Scrovegni Chapel, Padua",Bruce Cole,080761310X,4.47,1993,1,8,George Braziller,5:9,...,3:0,2:1,1:0,total:15,2,NaN,118,This beautiful series lavishly illustrates the...,2,"Giotto: The Scrovegni Chapel, Padua Bruce Cole..."
9,1000014,Haroun and the Sea of Stories,Salman Rushdie,0613495632,4.01,1991,1,11,Turtleback Books,5:10441,...,3:5842,2:1512,1:458,total:28578,64,eng,219,The author of The Satanic Verses returns with ...,64,Haroun and the Sea of Stories Salman Rushdie T...
12,1000027,Brilliant!: The Blinding Enlightenment of Niko...,Electric Company Theatre,097324819X,5.00,2004,4,11,Brindle and Glass Publishing,5:5,...,3:0,2:0,1:0,total:5,0,NaN,80,Chronicling the explosive career of a twentiet...,0,Brilliant!: The Blinding Enlightenment of Niko...
13,1000029,Feu Pâle,Vladimir Nabokov,2070383636,4.15,1991,1,4,Gallimard (folio),5:18954,...,3:5872,2:2002,1:1058,total:40099,1,fre,343,<p>The American poet John Shade is dead. His l...,1,Feu Pâle Vladimir Nabokov <p>The American poet...
14,1000030,Anne of Green Gables,L.M. Montgomery,1400100712,4.26,2003,1,8,Tantor Media,5:363709,...,3:97335,2:24252,1:12153,total:698753,6,NaN,0,When Marilla and Matthew Cuthbert of Green Gab...,6,Anne of Green Gables L.M. Montgomery When Mari...


In [28]:
book_data.loc[book_data['Description'].isnull()]

,Id,Name,Authors,ISBN,Rating,PublishYear,PublishMonth,PublishDay,Publisher,RatingDist5,...,RatingDist3,RatingDist2,RatingDist1,RatingDistTotal,CountsOfReview,Language,pagesNumber,Description,Count of text reviews,content


In [29]:
book_data.loc[book_data['Description'].isnull(), 'Description'] = ''

### Drop unnecessary columns

In [30]:
book_data = book_data.drop(columns=[
    'Id',
    'RatingDist5',
    'RatingDist4',
    'RatingDist3',
    'RatingDist2',
    'RatingDist1',
    'RatingDistTotal',
    'PublishMonth',
    'PublishDay',
    'Language'
])

In [31]:
book_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39705 entries, 0 to 39704
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Name                   39705 non-null  object 
 1   Authors                39705 non-null  object 
 2   ISBN                   39705 non-null  object 
 3   Rating                 39705 non-null  float64
 4   PublishYear            39705 non-null  int64  
 5   Publisher              39705 non-null  object 
 6   CountsOfReview         39705 non-null  int64  
 7   pagesNumber            39705 non-null  int64  
 8   Description            39705 non-null  object 
 9   Count of text reviews  39705 non-null  int64  
 10  content                39705 non-null  object 
dtypes: float64(1), int64(4), object(6)
memory usage: 3.3+ MB


In [32]:
book_data.isnull().sum()

Name                     0
Authors                  0
ISBN                     0
Rating                   0
PublishYear              0
Publisher                0
CountsOfReview           0
pagesNumber              0
Description              0
Count of text reviews    0
content                  0
dtype: int64

### ratings_data  processing (Collaborative Filtering )

In [33]:
ratings_data.isnull().sum()

user_id    0
book_id    0
rating     0
dtype: int64

In [34]:
ratings_data['user_id'] = ratings_data['user_id'].astype(str)
ratings_data['book_id'] = ratings_data['book_id'].astype(str)

In [35]:
ratings_data.duplicated().sum()

np.int64(1)

In [36]:
ratings_data = ratings_data.drop_duplicates()

In [37]:
user_item_matrix = ratings_data.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating'
)

In [38]:
user_item_matrix = user_item_matrix.fillna(0)
user_item_matrix.head()

book_id,B0001,B0002,B0003,B0004,B0005,B0006,B0007,B0008,B0009,B0010,...,B0191,B0192,B0193,B0194,B0195,B0196,B0197,B0198,B0199,B0200
user_id,,,,,,,,,,,,,,,,,,,,,
U0001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U0002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U0003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U0004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0
U0005,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0


### No need to drop anything already clean!!!

#### But if duplicates exist:

In [39]:
ratings_data = ratings_data.drop_duplicates()
ratings_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1033 entries, 0 to 1033
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  1033 non-null   object
 1   book_id  1033 non-null   object
 2   rating   1033 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 32.3+ KB


In [40]:
ratings_data.loc[:, ['user_id', 'book_id', 'rating']].head()

,user_id,book_id,rating
0,U0058,B0007,3
1,U0053,B0174,5
2,U0017,B0008,1
3,U0014,B0144,2
4,U0215,B0057,4


In [41]:
high_ratings = ratings_data.loc[ratings_data['rating'] >= 4]
high_ratings.head()

,user_id,book_id,rating
1,U0053,B0174,5
4,U0215,B0057,4
5,U0082,B0179,4
9,U0151,B0161,5
11,U0120,B0026,4


In [42]:
ratings_data.loc[ratings_data['user_id'] == 'U0001']

,user_id,book_id,rating
84,U0001,B0134,5
90,U0001,B0078,3
800,U0001,B0024,5


## This project builds a hybrid recommendation system for an online bookstore using:
- Collaborative Filtering (SVD-based matrix factorization)
- Content-Based Filtering (TF-IDF + Cosine Similarity)

The system generates personalized book recommendations to improve user engagement and experience.

## Problem Statement

An online bookstore wants to improve user experience by providing personalized book recommendations on its platform.

The system should analyze historical user-book interactions along with book metadata to generate meaningful suggestions.

The challenge is to design a **hybrid recommendation system** that:
- Uses collaborative filtering to learn user preferences from past ratings
- Uses content-based filtering to recommend similar books based on textual features such as title, author, and description
- Handles both existing users (with history) and new users (cold-start problem)
- Improves recommendation quality by combining both approaches

The final system should be able to return top-N recommended books for a given user efficiently.

#  Collaborative Filtering using Matrix Factorization (SVD)

This section implements collaborative filtering using Singular Value Decomposition (SVD).

The model learns latent relationships between users and books from the user-item interaction matrix.

**Import Required Libraries**   

In [43]:
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity


In [44]:
train_data, test_data = train_test_split(ratings_data,test_size=0.2,random_state=42)

In [51]:
train_matrix = train_data.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating'
)

In [52]:
user_item_matrix = train_data.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating'
).fillna(0)

In [53]:
from sklearn.metrics.pairwise import cosine_similarity
user_similarity = cosine_similarity(latent_matrix)

In [54]:
train_matrix_filled = train_matrix.fillna(0)

In [55]:
print(user_similarity.shape)

(282, 282)


## Applying Singular Value Decomposition (SVD)

Singular Value Decomposition (SVD) is a matrix factorization technique used in collaborative filtering recommendation systems.

The user-item interaction matrix contains many sparse and high-dimensional values. SVD reduces this large matrix into smaller latent feature representations while preserving important patterns between users and books.

By applying SVD:
- Hidden relationships between users and books can be discovered
- Dimensionality of the data is reduced
- Recommendation quality becomes more efficient and accurate
- Similar user preferences can be identified effectively

This helps the system generate personalized book recommendations based on historical user interactions.

In [56]:
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=50, random_state=42)
latent_matrix = svd.fit_transform(train_matrix_filled)

In [57]:
from sklearn.metrics.pairwise import cosine_similarity
user_similarity = cosine_similarity(latent_matrix)

In [ ]:
def get_collab_scores(user_id):

    user_index = user_item_matrix.index.get_loc(user_id)
    sim_scores = user_similarity[user_index]
    predicted_scores = np.dot(sim_scores, user_item_matrix.values)
    return predicted_scores

In [89]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=train_matrix.index,
    columns=train_matrix.index
)

In [90]:
def predict_ratings(user_id):
    user_index = train_matrix.index.get_loc(user_id)
    sim_scores = user_similarity[user_index]
    preds = np.dot(sim_scores, train_matrix_filled)
    norm = np.sum(np.abs(sim_scores))
    return pd.Series(preds / norm, index=train_matrix.columns)

### Recommended Function

In [91]:
def recommend_books(user_id, n=5):
    preds = predict_ratings(user_id)
    already_rated = train_matrix.loc[user_id]
    already_rated = already_rated[already_rated > 0].index
    recommendations = preds.drop(already_rated, errors='ignore')
    if recommendations.empty:
        return train_matrix.mean().sort_values(ascending=False).head(n)
    return recommendations.sort_values(ascending=False).head(n)

In [92]:
def get_reading_list(user_id, n=10):
    
    reading_list = {}
    
    # get recommendations
    top_n = recommend_books(user_id, n)
    
    for book_id, score in top_n.items():
        reading_list[book_id] = score
    
    return reading_list

In [93]:
example_reading_list = get_reading_list("U0058", 10)

print("\nTop Recommended Books\n")
print("-" * 40)

for book, score in example_reading_list.items():
    print(f"{book}: {score:.3f}")


Top Recommended Books

----------------------------------------
B0117: 0.311
B0174: 0.267
B0179: 0.249
B0048: 0.213
B0047: 0.155
B0152: 0.123
B0091: 0.115
B0056: 0.102
B0055: 0.098
B0186: 0.095


## Conclusion:

 - In this phase of the project, a collaborative filtering-based recommendation system was successfully implemented using Singular Value Decomposition (SVD). The model was trained on user–book rating interactions to learn hidden latent features that represent user preferences and item characteristics.

 - The system effectively transforms the sparse user–item rating matrix into a lower-dimensional latent space, allowing it to capture meaningful patterns in user behavior. By computing similarities in this latent space, the model is able to predict user preferences for unseen books.

 - The final output generates a ranked list of book recommendations for each user based on predicted preference scores. These results demonstrate that the model is capable of identifying relevant books that align with a user’s historical preferences.



##                             Content-Based Book Recommendation System

This module focuses on developing a content-based recommendation system for books. The primary objective is to recommend books based on their textual similarity rather than relying on user interaction patterns.

Unlike collaborative filtering, which depends on user–item rating behavior, this approach utilizes the intrinsic features of books, particularly their textual descriptions, to determine similarity between items.

## Objective

The objective of this component is to build a system capable of:

- Converting book descriptions into numerical feature representations  
- Measuring similarity between books using their content  
- Generating recommendations based on item-to-item similarity  
##  Expected Outcome

The model is expected to:

- Recommend books that are semantically similar in content  
- Work independently of user interaction data  
- Serve as a complementary approach to collaborative filtering in a hybrid recommendation system  

**Import Required Libraries**   

In [94]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import linear_kernel
import string

In [95]:
book_data.describe()

,Rating,PublishYear,CountsOfReview,pagesNumber,Count of text reviews
count,500.000000,500.000000,500.000000,500.00000,500.000000
mean,3.495780,1998.246000,11.270000,268.56800,11.270000
std,1.210831,8.574645,68.023905,188.85904,68.023905
min,0.000000,1959.000000,0.000000,0.00000,0.000000
25%,3.500000,1995.000000,0.000000,147.75000,0.000000
50%,3.860000,2000.000000,1.000000,235.50000,1.000000
75%,4.110000,2004.250000,4.000000,336.00000,4.000000
max,5.000000,2013.000000,1362.000000,1626.00000,1362.000000


In [96]:
book_data.nunique()

Name                     500
Authors                  413
ISBN                     499
Rating                   134
PublishYear               48
Publisher                370
CountsOfReview            58
pagesNumber              241
Description              439
Count of text reviews     58
content                  500
dtype: int64

In [97]:
book_data.isnull().sum()

Name                     0
Authors                  0
ISBN                     0
Rating                   0
PublishYear              0
Publisher                0
CountsOfReview           0
pagesNumber              0
Description              0
Count of text reviews    0
content                  0
dtype: int64

In [98]:
book_data['Name'] = book_data['Name'].fillna('')
book_data['Authors'] = book_data['Authors'].fillna('')
book_data['Description'] = book_data['Description'].fillna('')

In [99]:
book_data = book_data.head(500).copy()

In [100]:
book_data['content'] = (
    book_data['Name'] + ' ' +
    book_data['Authors'] + ' ' +
    book_data['Description']
)

In [101]:
for i, title in enumerate(book_data['Name'].dropna().head(20)):
    print(i, title)

0 Flight from Eden
1 Roommates Again
2 The King At The Door
3 Giotto: The Scrovegni Chapel, Padua
4 Larky Mavis
5 Computers, Chess and Long-Range Planning
6 The Holy Longing
7 Yearning to Breathe Free: Robert Smalls of South Carolina and His Families
8 A Wild Yearning
9 Haroun and the Sea of Stories
10 There and Back Again: An Actor's Tale
11 The Desire and Pursuit of the Whole: A Romance of Modern Venice
12 Brilliant!: The Blinding Enlightenment of Nikola Tesla
13 Feu Pâle
14 Anne of Green Gables
15 Getting Home Alive
16 Getting Home Alive! Memories of Car Collecting Before It Got Civilized
17 Turning Custom Duck and Game Calls: The Complete Guide for Craftsmen, Collectors, and Outdoorsmen
18 Behind the Secret Window
19 In the Sewers of Lvov: A Heroic Story of Survival from the Holocaust


In [102]:
toprated = book_data.sort_values(by='Rating', ascending=False).head(10)

In [103]:
popular = book_data.sort_values(by='CountsOfReview', ascending=False).head(10)

In [104]:
print("Top 10 rated books are -")
print("")

for i in toprated['Name']:
    print(i)

print("")
print("---")
print("")

print("Top 10 popular books are -")
print("")

for j in popular['Name']:
    print(j)

Top 10 rated books are -

The Gift in Every Day: Little Lessons on Living a Big Life
El Viaje de Colon
Cucina Rapida: Quick Italian-Style Home Cooking
My Books of Vehicles Slipcase Box Set
Concepts Of Epidemiology: An Integrated Introduction To The Ideas, Theories, Principles And Methods Of Epidemiology
Brilliant!: The Blinding Enlightenment of Nikola Tesla
Master of None: An Autobiography
Ceramic Coin Banks: Identification & Value Guide
Hidden In Plain View: Refugees Living Without Protection In Nairobi And Kampala
Age Is Nothing: Attitude Is Everything

---

Top 10 popular books are -

Pollyanna (Pollyanna, #1)
Phineas Gage: A Gruesome but True Story About Brain Science
Jackie & Me (A Baseball Card Adventure, #2)
Honus & Me (A Baseball Card Adventure, #1)
My Secret War: The World War II Diary of Madeline Beck, Long Island, New York 1941 (Dear America)
Rex Libris, Volume I: I, Librarian (Rex Libris, #1-5)
Just David
Getting Air
I Kissed the Baby!
Naruto, Vol. 16: Eulogy (Naruto, #16)


### Create mapping

In [105]:
Map = dict(enumerate(book_data['ISBN']))
revMap = {v: k for k, v in Map.items()}

### CREATE TF-IDF MATRIX


In [106]:
book_data = book_data.reset_index(drop=True)

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=3000
)
tfidf_matrix = tfidf.fit_transform(book_data['content'])

### COSINE SIMILARITY MATRIX

In [107]:
def printDetails(book_index):

    print("Title:",
          book_data.iloc[book_index]['Name'])
    print("Author:",
          book_data.iloc[book_index]['Authors'])
    print("Publication Year:",
          book_data.iloc[book_index]['PublishYear'])
    print("Publisher:",
          book_data.iloc[book_index]['Publisher'])
    print("Rating:",
          book_data.iloc[book_index]['Rating'])
    print("ISBN:",
          book_data.iloc[book_index]['ISBN'])
    print(" ")

In [108]:
def getRecommendations(book_index, top_n=10):

    # similarity for ONLY one book
    sim_scores = cosine_similarity(
        tfidf_matrix[book_index],
        tfidf_matrix
    ).flatten()

    # top similar books
    top_indices = sim_scores.argsort()[-top_n-1:-1][::-1]

    print(" ")
    print("+----------------------+")
    print("Books you might like")
    print("+----------------------+")
    print(" ")

    for i in top_indices:
        printDetails(i)

In [109]:
def newRecommendations():

    print("")
    print("+-------------------------------+")
    print("Popular books to start from")
    print("+-------------------------------+")
    print("")

    for j in popular['Name']:
        print(j)


In [110]:
for i, title in enumerate(book_data['Name'].head(20)):
    print(i, "-", title)
getRecommendations(0)

0 - Flight from Eden
1 - Roommates Again
2 - The King At The Door
3 - Giotto: The Scrovegni Chapel, Padua
4 - Larky Mavis
5 - Computers, Chess and Long-Range Planning
6 - The Holy Longing
7 - Yearning to Breathe Free: Robert Smalls of South Carolina and His Families
8 - A Wild Yearning
9 - Haroun and the Sea of Stories
10 - There and Back Again: An Actor's Tale
11 - The Desire and Pursuit of the Whole: A Romance of Modern Venice
12 - Brilliant!: The Blinding Enlightenment of Nikola Tesla
13 - Feu Pâle
14 - Anne of Green Gables
15 - Getting Home Alive
16 - Getting Home Alive! Memories of Car Collecting Before It Got Civilized
17 - Turning Custom Duck and Game Calls: The Complete Guide for Craftsmen, Collectors, and Outdoorsmen
18 - Behind the Secret Window
19 - In the Sewers of Lvov: A Heroic Story of Survival from the Holocaust
 
+----------------------+
Books you might like
+----------------------+
 
Title: The Anarchist Collectives: Workers' Self-Management in the Spanish Revolution 

## Hybrid Recommendation System

In this project, a hybrid recommendation system has been implemented to improve recommendation quality by combining the strengths of both content-based filtering and collaborative filtering approaches.

### 1. Content-Based Filtering
The content-based model uses TF-IDF (Term Frequency–Inverse Document Frequency) to convert book metadata such as title, author, and description into numerical feature vectors. Cosine similarity is then computed between books to identify items that are similar in terms of textual content. This approach helps in recommending books that are similar to a user's previously liked items.

### 2. Collaborative Filtering
The collaborative filtering model is built using matrix factorization with Truncated Singular Value Decomposition (SVD). A user-item interaction matrix is created from historical ratings data. SVD decomposes this matrix into latent features representing hidden preferences of users and characteristics of books. Recommendations are generated based on similarity in this latent feature space.

### 3. Hybrid Approach
To improve recommendation accuracy and overcome limitations of individual models, a hybrid approach is used. The final recommendation score is calculated using a weighted combination of both models:

Final Score = α × Content Score + (1 − α) × Collaborative Score

Where:
- Content Score is derived from cosine similarity between TF-IDF vectors
- Collaborative Score is derived from SVD-based user preference predictions
- α is a tunable parameter that controls the contribution of each model

### 4. Advantages of Hybrid System
- Improves recommendation accuracy by combining two approaches
- Handles both new and existing users effectively
- Reduces cold-start problem using content-based filtering
- Captures both textual similarity and user behavior patterns

### 5. Outcome
The hybrid system provides more relevant and personalized book recommendations compared to using only a single filtering method.

In [112]:
book_data = book_data.reset_index(drop=True)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def hybrid_recommend(user_id, book_index, top_n=10, alpha=0.6):

    content_scores = cosine_similarity(
        tfidf_matrix[book_index],
        tfidf_matrix
    ).flatten()

    collab_scores = get_collab_scores(user_id)
    collab_scores = np.array(collab_scores)
    min_len = min(len(content_scores), len(collab_scores))

    content_scores = content_scores[:min_len]
    collab_scores = collab_scores[:min_len]

    hybrid_scores = (alpha * content_scores) + ((1 - alpha) * collab_scores)

    hybrid_scores[book_index] = -1

    top_indices = hybrid_scores.argsort()[-top_n:][::-1]

    return book_data.iloc[top_indices][['Name', 'Authors', 'Rating']]

In [115]:
hybrid_recommend("U0058", 0)

,Name,Authors,Rating
73,A Season on the Mat: Dan Gable and the Pursuit...,Nolan Zavoral,4.19
6,The Holy Longing,Connie Zweig,3.83
113,Space Wars: Worlds & Weapons,Steven Eisler,3.93
170,Count Zinzendorf: Firstfruit,Janet Benge,3.90
175,Dictionnaire des symboles,Jean Chevalier,4.33
46,"Strontium Dog: Search/Destroy Agency Files, Vo...",John Wagner,4.03
45,At the Plate with...Ichiro,Matt Christopher,4.29
148,"The Steel, the Mist, and the Blazing Sun",Christopher Anvil,3.18
88,The Lighthouse at the End of the World,Stephen Marlowe,3.53
53,Tristan And Iseult,Rosemary Sutcliff,3.62
